# 08 — XGBoost Demand Forecasting

## Purpose
Train an **XGBoost regression model** for each of the 35 (branch × product) pairs
to predict the **daily quantity sold** of each top-5 product at each branch.

**Why XGBoost for this problem?**
- Handles tabular data with mixed feature types (numeric lags, encoded categoricals)
- Naturally captures non-linear interactions (e.g., temperature × day of week)
- Robust to outliers — a single spike sale won't derail the model
- Rolling window features give it the auto-regressive signal time-series models depend on
- Fast training — 35 models run in seconds

## Modeling approach
- **One model per (branch, product) pair** — 35 models total (7 branches × 5 products)
- **Chronological train/validation split** — last 90 days held out for evaluation
- **Early stopping** — training stops automatically when validation RMSE plateaus
- **Features:** 9 rolling lags + temporal + weather + holiday (18 total)
- **Target:** `quantity` — units sold that day

## Input
`data/processed/top5_per_branch.csv` — produced by `07_top_products.ipynb`

## Run order
Run after `07_top_products.ipynb`.

In [ ]:
import os

# ─── CHANGE THIS FLAG ─────────────────────────────────────────────────
USE_GITHUB  = False   # True = read from GitHub (Colab), False = read from local clone
# ───────────────────────────────────────────────────────────────────────

GITHUB_BASE = "https://media.githubusercontent.com/media/DiegoLarrieta/PanemReto/main"

if USE_GITHUB:
    PROCESSED_DIR = f"{GITHUB_BASE}/data/processed"
else:
    # This notebook lives in models/xgboost/ — go up two levels to reach project root
    PROJECT_ROOT  = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
    PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

print("PROCESSED_DIR:", PROCESSED_DIR)

## Step 1 — Imports and data load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

# ── Load ──────────────────────────────────────────────────────────────
if USE_GITHUB:
    input_path = f"{PROCESSED_DIR}/top5_per_branch.csv"
else:
    input_path = os.path.join(PROCESSED_DIR, "top5_per_branch.csv")

df = pd.read_csv(input_path, low_memory=False)
df["operating_date"] = pd.to_datetime(df["operating_date"])
df["quantity"]       = pd.to_numeric(df["quantity"], errors="coerce").fillna(0)

print(f"Loaded: {len(df):,} rows")
print(f"Date range: {df['operating_date'].min().date()} → {df['operating_date'].max().date()}")
print(f"\nRows per branch:")
print(df.groupby("sucursal")["item"].count().to_string())
print(f"\nProducts per branch:")
print(df.groupby("sucursal")["item"].nunique().to_string())

## Step 2 — Feature preparation

XGBoost requires all inputs to be numeric. We encode the three categorical columns:

| Column | Original type | Encoding | Reason |
|--------|--------------|----------|--------|
| `day_name` | Spanish strings | Integer 0 (lunes) → 6 (domingo) | Ordinal — captures day-of-week cycle |
| `cold_or_warm` | 'cold' / 'warm' | Binary: warm=1, cold=0 | Already binary — direct flag |
| `holiday_type` | 3 categories | Ordinal: No holiday=0, Puente=1, Festivo Oficial=2 | Captures severity of holiday |
| `holidays` | bool | Already 0/1 — kept as `is_holiday` | No change needed |

We also derive `day_of_year` and `year` from `operating_date`:
- `day_of_year` (1–365) helps the model learn yearly seasonal patterns
- `year` lets the model capture multi-year trends (growth, decline)

**Final feature set: 18 columns**
- 9 rolling lags (qty_roll_1 → qty_roll_365)
- 5 temporal (day_of_week, week_number, month, day_of_year, year)
- 2 weather (tavg, is_warm)
- 2 holiday (is_holiday, holiday_type_enc)

In [ ]:
# ── Day of week (Spanish → integer 0–6) ──────────────────────────────
day_map = {
    "lunes": 0, "martes": 1, "mi\u00e9rcoles": 2, "miercoles": 2,
    "jueves": 3, "viernes": 4, "s\u00e1bado": 5, "sabado": 5, "domingo": 6
}
df["day_of_week"] = df["day_name"].str.lower().map(day_map)

# ── Temperature (binary) ──────────────────────────────────────────────
df["is_warm"] = (df["cold_or_warm"] == "warm").astype(int)

# ── Holiday type (ordinal) ────────────────────────────────────────────
holiday_type_map = {"No holiday": 0, "Puente": 1, "Festivo Oficial": 2}
df["holiday_type_enc"] = df["holiday_type"].map(holiday_type_map).fillna(0).astype(int)

# ── Holiday flag ──────────────────────────────────────────────────────
df["is_holiday"] = df["holidays"].astype(int)

# ── Additional temporal features ──────────────────────────────────────
df["day_of_year"] = df["operating_date"].dt.dayofyear
df["year"]        = df["operating_date"].dt.year

# ── Final feature list ────────────────────────────────────────────────
FEATURE_COLS = [
    # Auto-regressive — lagged demand signals (most important group)
    "qty_roll_1",   # units sold yesterday
    "qty_roll_3",   # sum last 3 days
    "qty_roll_7",   # sum last 7 days (weekly cycle)
    "qty_roll_15",  # sum last 15 days
    "qty_roll_30",  # sum last 30 days (monthly)
    "qty_roll_60",  # sum last 60 days
    "qty_roll_90",  # sum last 90 days (quarterly)
    "qty_roll_180", # sum last 180 days (semi-annual)
    "qty_roll_365", # sum last 365 days (year-over-year)
    # Temporal
    "day_of_week",  # 0=Monday … 6=Sunday
    "week_number",  # ISO week (1–53)
    "month",        # 1–12
    "day_of_year",  # 1–365
    "year",         # captures multi-year trend
    # Weather
    "tavg",         # average temperature °C
    "is_warm",      # 1 if tavg >= 25°C
    # Holidays
    "is_holiday",       # 1 if any holiday
    "holiday_type_enc", # 0=none, 1=Puente, 2=Festivo Oficial
]
TARGET_COL = "quantity"

# Verify
missing = [c for c in FEATURE_COLS if c not in df.columns]
if missing:
    print(f"WARNING — missing columns: {missing}")
else:
    print(f"All {len(FEATURE_COLS)} feature columns present.")
    print(f"\nFeature groups:")
    print(f"  Rolling lags  : {[c for c in FEATURE_COLS if 'roll' in c]}")
    print(f"  Temporal      : {[c for c in FEATURE_COLS if c in ['day_of_week','week_number','month','day_of_year','year']]}")
    print(f"  Weather       : {[c for c in FEATURE_COLS if c in ['tavg','is_warm']]}")
    print(f"  Holidays      : {[c for c in FEATURE_COLS if 'holiday' in c]}")

## Step 3 — Train / Validation Split

**Rule: chronological only — never shuffle time-series data.**

Shuffling would let the model train on future data to predict the past — the validation
score would look great but performance in production would be far worse.

For each (branch, product) pair:
1. Sort all rows by `operating_date`
2. **Validation** = last `VAL_DAYS = 90` calendar days
3. **Training** = everything before that

```
Timeline:
│──────────────── TRAIN ──────────────────│── VAL (90d) ──│
2022-01-01                         cutoff          2026-02-12
```

**Why 90 days?**
- Long enough to cover ~13 full weekly cycles — sufficient to measure weekly pattern accuracy
- Long enough to include at least one or two holidays
- Aligns with a business quarter — a meaningful deployment window

> **Note on shorter branches:** Credi Club (~448 days) and Plaza Nativa (~722 days)
> have less history than the others. For these, 90 days is still a valid validation
> window, but training data is limited. We print a warning when training days < 180.

## Step 4 — XGBoost Parameters

We use one conservative parameter set across all 35 models. Conservative means:
moderate depth, low learning rate, and regularization — this reduces overfit risk
on the smaller per-product datasets.

| Parameter | Value | What it controls |
|-----------|-------|------------------|
| `n_estimators` | 1000 | Max number of trees to build. Early stopping will stop before this. |
| `max_depth` | 5 | Max depth of each tree. 5 allows moderately complex patterns without overfitting. |
| `learning_rate` | 0.05 | How much each new tree corrects the previous error. Lower = slower but more robust. |
| `subsample` | 0.8 | Fraction of training rows sampled per tree. Adds randomness, reduces variance. |
| `colsample_bytree` | 0.8 | Fraction of features sampled per tree. Same rationale as subsample. |
| `min_child_weight` | 3 | Minimum sum of instance weights in a leaf node. Prevents very specific leaf splits. |
| `reg_alpha` | 0.1 | L1 regularization on leaf weights. Encourages sparsity — unimportant features → 0. |
| `reg_lambda` | 1.0 | L2 regularization on leaf weights. Penalizes large weights — smooths the model. |
| `objective` | reg:squarederror | Loss function: minimize squared error. Standard for regression. |
| `eval_metric` | rmse | Metric used by early stopping to decide when to stop training. |
| `early_stopping_rounds` | 50 | Stop if validation RMSE hasn't improved in 50 consecutive rounds. |
| `random_state` | 42 | Reproducibility seed — same results every run. |

**Early stopping** is the most important parameter here:
instead of always building 1000 trees, we stop the moment the model stops
improving on the validation set. This prevents overfitting automatically
and speeds up training. The `best_iter` column in the results shows how many
trees each model actually used.

In [ ]:
VAL_DAYS = 90   # calendar days held out for validation

XGB_PARAMS = dict(
    n_estimators          = 1000,
    max_depth             = 5,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    min_child_weight      = 3,
    reg_alpha             = 0.1,   # L1
    reg_lambda            = 1.0,   # L2
    objective             = "reg:squarederror",
    eval_metric           = "rmse",
    early_stopping_rounds = 50,
    random_state          = 42,
    n_jobs                = -1,
    verbosity             = 0,     # suppress XGBoost output
)

print("XGBoost parameters:")
for k, v in XGB_PARAMS.items():
    print(f"  {k:<25} = {v}")

## Step 5 — Training Loop

We iterate over every (branch, product) pair and:
1. Filter data to that pair, sort by date, drop rows with NaN features
   *(the first few rows after a branch opens have no rolling history yet)*
2. Split chronologically into train / validation
3. Train XGBoost with early stopping on the validation set
4. Predict on the validation set, clip negatives to 0
5. Compute 4 evaluation metrics:

| Metric | Formula | Best for |
|--------|---------|----------|
| **MAE** | mean(\|y − ŷ\|) | Most interpretable — error in actual units |
| **RMSE** | √mean((y − ŷ)²) | Penalizes large single-day errors more than MAE |
| **MAPE** | mean(\|y − ŷ\| / y) × 100 | % error — computed only on days where y > 0 |
| **WAPE** | sum(\|y − ŷ\|) / sum(y) × 100 | Weighted % error — robust when zero-sale days exist |

**WAPE is our primary metric** because many products have zero-sale days in the
validation window (closed days, holidays, stockouts). MAPE is undefined on those days
(division by zero), while WAPE handles them gracefully by weighting each day by its
actual demand.

In [ ]:
results        = []   # one dict per trained model
trained_models = {}   # (branch, product) → (model, val_df, y_pred)

# Build full list of (branch, product) pairs
pairs = [
    (branch, product)
    for branch in sorted(df["sucursal"].dropna().unique())
    for product in sorted(df[df["sucursal"] == branch]["item"].unique())
]

print(f"Training {len(pairs)} models (one per branch × product)...\n")
print(f"{'Branch':<35} {'Product':<35} {'MAE':>6} {'RMSE':>6} {'WAPE':>7} {'Iter':>5}")
print("-" * 100)

for branch, product in pairs:
    # ── Filter & sort ──────────────────────────────────────────────
    sub = (
        df[(df["sucursal"] == branch) & (df["item"] == product)]
        .sort_values("operating_date")
        .dropna(subset=FEATURE_COLS + [TARGET_COL])
        .copy()
    )

    if len(sub) < VAL_DAYS + 30:
        print(f"  SKIP  {branch} | {product} — only {len(sub)} rows after dropna")
        continue

    # ── Chronological split ────────────────────────────────────────
    cutoff = sub["operating_date"].max() - pd.Timedelta(days=VAL_DAYS)
    train  = sub[sub["operating_date"] <= cutoff]
    val    = sub[sub["operating_date"] >  cutoff]

    if len(train) < 180:
        print(f"  WARN  {branch} | {product} — only {len(train)} training days (< 180)")

    X_train = train[FEATURE_COLS].values
    y_train = train[TARGET_COL].values
    X_val   = val[FEATURE_COLS].values
    y_val   = val[TARGET_COL].values

    # ── Train ──────────────────────────────────────────────────────
    model = XGBRegressor(**XGB_PARAMS)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    # ── Predict — clip negatives (demand can't be negative) ────────
    y_pred = np.clip(model.predict(X_val), 0, None)

    # ── Metrics ────────────────────────────────────────────────────
    mae  = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))

    # MAPE — skip zero-actual days
    nonzero = y_val > 0
    mape = (
        (np.abs(y_val[nonzero] - y_pred[nonzero]) / y_val[nonzero]).mean() * 100
        if nonzero.sum() > 0 else np.nan
    )

    # WAPE — weighted, handles zero days gracefully
    wape = (
        np.abs(y_val - y_pred).sum() / y_val.sum() * 100
        if y_val.sum() > 0 else np.nan
    )

    results.append({
        "branch"     : branch,
        "product"    : product,
        "train_days" : len(train),
        "val_days"   : len(val),
        "best_iter"  : model.best_iteration,
        "mae"        : round(mae,  2),
        "rmse"       : round(rmse, 2),
        "mape"       : round(mape, 1) if not np.isnan(mape) else np.nan,
        "wape"       : round(wape, 1) if not np.isnan(wape) else np.nan,
    })
    trained_models[(branch, product)] = (model, val, y_pred)

    print(f"  {branch:<35} {product:<35} {mae:>6.1f} {rmse:>6.1f} {wape:>6.1f}% {model.best_iteration:>5}")

print(f"\nDone. {len(results)} models trained.")

## Step 6 — Results Summary

The table below collects all 4 metrics for every (branch, product) pair.

**How to read the metrics:**
- `mae` is in the same units as `quantity` — an MAE of 5.0 means predictions are off by 5 units on average
- `rmse` > `mae` means there were some large single-day errors (spikes)
- `wape` is the primary metric:
  - **< 20%** → Good — usable for daily stock planning as-is
  - **20–40%** → Acceptable — directionally correct, add a safety stock buffer
  - **> 40%** → Poor — model needs tuning or product has too little history
- `best_iter` shows how many trees early stopping kept (out of the 1000 maximum)

In [ ]:
results_df = pd.DataFrame(results)

# ── Full results table ────────────────────────────────────────────────
print("=== All models — evaluation metrics ===\n")
display(
    results_df[["branch", "product", "train_days", "val_days", "best_iter",
                "mae", "rmse", "mape", "wape"]]
    .sort_values(["branch", "wape"])
    .reset_index(drop=True)
)

# ── Aggregate stats across all models ────────────────────────────────
print("\n=== Aggregate statistics (all 35 models) ===")
display(
    results_df[["mae", "rmse", "mape", "wape"]]
    .agg(["mean", "median", "min", "max"])
    .round(1)
)

# ── Per-branch summary ────────────────────────────────────────────────
print("\n=== Average MAE and WAPE per branch ===")
display(
    results_df.groupby("branch")[["mae", "rmse", "wape"]]
    .mean().round(1)
    .sort_values("wape")
)

# ── WAPE distribution ─────────────────────────────────────────────────
good     = (results_df["wape"] < 20).sum()
ok       = ((results_df["wape"] >= 20) & (results_df["wape"] < 40)).sum()
poor     = (results_df["wape"] >= 40).sum()
print(f"\n=== WAPE distribution ===")
print(f"  Good   (WAPE < 20%)  : {good} models")
print(f"  OK     (20% – 40%)   : {ok} models")
print(f"  Poor   (WAPE > 40%)  : {poor} models")

## Step 7 — Feature Importance

XGBoost computes **feature importance** as the average `gain` each feature contributes
when it is chosen as a split criterion across all trees.

We aggregate importances across all 35 models to get a global picture of
which features matter most for this forecasting task.

**Expected pattern:**
- `qty_roll_1` and `qty_roll_7` should dominate — yesterday's sales and last week's
  trend are the strongest predictors of tomorrow
- `qty_roll_365` captures year-over-year seasonality
- `day_of_week` and `month` capture cyclical patterns
- `tavg` / `is_warm` show how weather-sensitive demand is
- Holiday features matter most for branches near hospitals and hotels
  (different customer mix on holidays)

In [ ]:
# Collect gain-based importance from every trained model
importance_rows = []
for (branch, product), (model, val, y_pred) in trained_models.items():
    imp = model.get_booster().get_score(importance_type="gain")
    row = {feat: imp.get(feat, 0.0) for feat in FEATURE_COLS}
    row["branch"]  = branch
    row["product"] = product
    importance_rows.append(row)

imp_df = pd.DataFrame(importance_rows)
mean_imp = imp_df[FEATURE_COLS].mean().sort_values(ascending=False)

# ── Bar chart ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 6))
colors = ["#169830" if i < 5 else "#a8d5b0" for i in range(len(mean_imp))]
ax.barh(mean_imp.index[::-1], mean_imp.values[::-1], color=colors[::-1])
ax.set_xlabel("Mean Gain (averaged across all 35 models)")
ax.set_title("Feature Importance — XGBoost\n(Aggregated Across All Branch × Product Models)")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

# ── Table ─────────────────────────────────────────────────────────────
print("\nFeature importance ranking (mean gain):")
print(mean_imp.round(1).to_string())

## Step 8 — Actual vs Predicted (Validation Period)

For each branch we pick the **best model** (lowest WAPE) and plot actual vs predicted
quantity over the 90-day validation window.

**What to look for:**
- The predicted line (green dashed) should follow the actual line (black) closely
- Weekly oscillations should be captured — if the model flattens them out,
  `day_of_week` importance is too low
- No systematic bias — predictions should be above the actual line as often as below
- Spikes (e.g., from a promo or event) will likely be missed — that is expected
  because the model has no event data as input

In [ ]:
# Best model per branch = lowest WAPE
best_per_branch = (
    results_df.sort_values("wape")
              .groupby("branch", as_index=False)
              .first()[["branch", "product", "mae", "rmse", "mape", "wape"]]
)

print("Best model per branch (by WAPE):")
display(best_per_branch)

n = len(best_per_branch)
fig, axes = plt.subplots(n, 1, figsize=(14, 4 * n))
if n == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, best_per_branch.iterrows()):
    branch, product = row["branch"], row["product"]
    model, val, y_pred = trained_models[(branch, product)]

    ax.plot(val["operating_date"].values, val[TARGET_COL].values,
            label="Actual", color="#222222", linewidth=1.5)
    ax.plot(val["operating_date"].values, y_pred,
            label="Predicted", color="#169830", linewidth=1.5, linestyle="--")

    ax.fill_between(
        val["operating_date"].values,
        val[TARGET_COL].values, y_pred,
        alpha=0.12, color="#169830", label="Error"
    )

    ax.set_title(
        f"{branch}  |  {product}\n"
        f"MAE={row['mae']:.1f} units   RMSE={row['rmse']:.1f}   WAPE={row['wape']:.1f}%",
        fontsize=9
    )
    ax.set_ylabel("Units sold")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")

plt.suptitle("Actual vs Predicted — Validation Period (90 days)\nBest model per branch by WAPE",
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

## Step 9 — Worst Models (Diagnosis)

It is equally important to look at the **worst performing models** to understand
where XGBoost struggles. Common causes:
- Too little training data (Credi Club)
- Very irregular demand (a product that sells in batches once a week)
- A product that was introduced recently (no rolling history for the 180/365-day lags)

We plot the worst model per branch (highest WAPE).

In [ ]:
# Worst model per branch = highest WAPE
worst_per_branch = (
    results_df.sort_values("wape", ascending=False)
              .groupby("branch", as_index=False)
              .first()[["branch", "product", "mae", "rmse", "mape", "wape"]]
)

print("Worst model per branch (by WAPE):")
display(worst_per_branch)

n = len(worst_per_branch)
fig, axes = plt.subplots(n, 1, figsize=(14, 4 * n))
if n == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, worst_per_branch.iterrows()):
    branch, product = row["branch"], row["product"]
    if (branch, product) not in trained_models:
        continue
    model, val, y_pred = trained_models[(branch, product)]

    ax.plot(val["operating_date"].values, val[TARGET_COL].values,
            label="Actual", color="#222222", linewidth=1.5)
    ax.plot(val["operating_date"].values, y_pred,
            label="Predicted", color="#d44000", linewidth=1.5, linestyle="--")

    ax.fill_between(
        val["operating_date"].values,
        val[TARGET_COL].values, y_pred,
        alpha=0.12, color="#d44000", label="Error"
    )

    ax.set_title(
        f"{branch}  |  {product}\n"
        f"MAE={row['mae']:.1f} units   RMSE={row['rmse']:.1f}   WAPE={row['wape']:.1f}%",
        fontsize=9
    )
    ax.set_ylabel("Units sold")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")

plt.suptitle("Actual vs Predicted — Validation Period (90 days)\nWorst model per branch by WAPE",
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

## Summary

| | |
|---|---|
| Models trained | 35 (one per branch × product) |
| Features | 18 (9 rolling lags + 5 temporal + 2 weather + 2 holiday) |
| Validation window | Last 90 days (chronological) |
| Primary metric | WAPE — robust to zero-sale days |
| Training strategy | Early stopping (up to 1000 trees, stop at 50 rounds no improvement) |

### Interpreting WAPE results
| WAPE | Interpretation | Recommended action |
|------|---------------|--------------------|
| < 20% | Good | Use predictions directly for stock planning |
| 20–40% | Acceptable | Add a 20–30% safety stock buffer on top |
| > 40% | Poor | Investigate data quality; consider Prophet or ARIMA for that product |

### Next steps
1. **Hyperparameter tuning** — use `TimeSeriesSplit` cross-validation to search
   `max_depth`, `learning_rate`, and `subsample` per product group
2. **Prophet model** → `models/prophet/` — compare on the same 90-day validation window
3. **ARIMA baseline** → `models/arima/` — verify XGBoost beats the univariate baseline
4. **Ensemble** — average XGBoost + Prophet predictions for production deployment